# Phase 15C — Docker serving integration

ProductIQ now serves **real** search, product details, and recommendations from Docker Compose when production serving is enabled and catalog data has been loaded **explicitly**.

Canonical architecture notes: `docs/architecture/docker-serving-integration.md`.

This notebook is documentation, not a training run. It does not download models or load the 367,172-row catalog.

## Why this is not health-only

Phase 15A proved the API process starts. Phase 15B proved PostgreSQL/pgvector schema. Phase 15C connects those to production pipelines that already existed:

1. `products` + `embedding vector(384)` + HNSW cosine index
2. BM25 lexical retrieval (persisted pickle, or an in-memory index built from labeled fixture rows)
3. BGE query encoding + pgvector cosine similarity
4. Reciprocal Rank Fusion hybrid retrieval
5. PostgreSQL hard filters and **baseline** ranking (`retrieve_ranked`; experimental LightGBM LTR is not the HTTP default)
6. `RecommendationPipeline` (vector + attribute + BM25 generators, seed exclusion, diversity)
7. Product lookup by `product_id` from PostgreSQL

Large artifacts stay on a **read-only host mount**. The API image does not bake the catalog parquet, BM25 pickle, or embedding parquet.

## Data flow

```text
resources/processed/product_catalog.parquet
        |  explicit operator command (never ordinary API startup)
        v
PostgreSQL products  --ON CONFLICT product_id upsert, chunk size 10_000-->

resources/processed/product_embeddings.parquet
        |  explicit operator command (reuse validated 384-d artifact)
        v
products.embedding vector(384) + ix_products_embedding_hnsw_cosine

Query POST /api/v1/search
  -> QueryRepresentation (+ optional hard filters)
  -> BM25Retriever + SemanticRetriever(pgvector)
  -> RRF hybrid pool
  -> PostgresCatalogCandidateFilter
  -> baseline ranking (not LTR)
  -> SearchApiResponse {product_id, rank} only
```

## Modules reused (not rewritten)

| Concern | Module |
|---------|--------|
| Catalog upsert | `productiq.database.catalog_pipeline.load_processed_catalog` |
| Embedding load + HNSW | `productiq.retrieval.vector_index_builder.build_product_vector_index` |
| Schema bootstrap | `productiq.database.bootstrap_compose` |
| BM25 | `create_bm25_retriever_from_index_path` / in-memory fixture index |
| Semantic | `create_semantic_retriever_from_engine` |
| Hybrid | `create_rrf_hybrid_retriever` + `ProductionRetrievalPipeline.retrieve_ranked` |
| Recommendations | `build_production_recommendation_pipeline` |
| Product read | `PostgresProductCatalogReadProvider` |
| HTTP wiring | existing search/product/recommendation serving services |

New glue lives in `load_runtime_catalog`, `production_stack`, and Compose `catalog-load`.

## Explicit load (never at import or ordinary startup)

Fixture mode inserts three labeled `compose-demo-*` rows. That is **not** the AJIO catalog.

```bash
docker compose up -d postgres
docker compose --profile load run --rm catalog-load
docker compose up -d productiq-api
```

Full catalog (operator-driven, long-running, not claimed unless row counts are inspected):

```bash
python -m productiq.database.load_runtime_catalog --mode full --enforce-official-checksums --json
```

Restartability: catalog upserts `ON CONFLICT (product_id)`; embeddings `UPDATE` by `product_id`. Repeat fixture load is safe (SQLAlchemy `merge`).

## Lifecycle and readiness

- `from productiq.api.app import create_app` does not open PostgreSQL.
- `PRODUCTIQ_ENABLE_PRODUCTION_SERVING` defaults to false so unit tests stay on the hollow app.
- Compose sets the flag to `1` for `productiq-api` only.
- SQLAlchemy sessions are opened per request for catalog filter, ranking context, product get, and recommendations.
- Lifespan disposes the engine on shutdown.
- `/api/v1/ready` adds `catalog_data` with mode `fixture` / `full` / `partial` / `empty`.
- A live database connection does **not** imply search is ready; search/recommendation/product must be wired, and catalog mode is reported separately.

First API start with serving enabled downloads BGE into the `productiq_hf_cache` volume. That is model load, not catalog load.

## HTTP contracts exercised

| Route | Behavior |
|-------|----------|
| `POST /api/v1/search` | Real BM25 + pgvector + RRF; public items are `product_id` + `rank` |
| `GET /api/v1/products/{id}` | PostgreSQL lookup; 404 envelope for unknown IDs |
| `POST /api/v1/recommendations` | Seed from catalog; seed excluded; established recommendation score field |
| `GET /api/v1/ready` | Distinguishes configured services from catalog fixture vs full |

Opt-in integration tests: `PRODUCTIQ_PHASE_15C_INTEGRATION=1 pytest tests/serving/integration -m integration`.

## Interview notes

- Serving is composition, not a new retrieval stack. The HTTP adapter wraps `ProductionRetrievalPipeline` and `RecommendationPipeline`.
- Experimental LTR stays behind `experimental_ltr` on `ProductionRankingStageConfig`. HTTP search calls `retrieve_ranked`, which is asserted to use the baseline path.
- Fixture vs full is a **labeling** problem as much as a loading problem. Readiness text explicitly says the three-row demo is not the 367,172-row catalog.
- Embedding generation is not repeated at serving time when `product_embeddings.parquet` already exists. The loader writes vectors into pgvector; the API encodes **queries** with BGE.
- PostgreSQL port 5432 stays on the Compose network. Only the API port is published.

## Known limits

- Full 367k load is supported by the CLI but is not part of `docker compose up`.
- Fixture embeddings are 384-d unit vectors. Lexical search is the reliable fixture path unless a real encoder is used against BGE catalog embeddings.
- Existing vector loader updates embeddings per row inside batched Parquet reads; a full load can take a long time.
- Next.js, cloud deploy, auth, Redis, and Kubernetes are later phases.